# 🥉 Bronze Layer
Reading JSON files and writing to Delta table.

## Change the path to import modules from the parent directory

In [ ]:
import sys
sys.path.append("..")

## Importing necessary libraries

In [ ]:
from spark_session import get_spark
from config import get_config
from pyspark.sql.types import *
import pandas as pd
import glob

## Detect environment

In [ ]:
cfg = get_config()

## Change settings to display all columns in the DataFrame

In [ ]:
pd.set_option("display.max_columns", None)

## Intialize Spark session

In [ ]:
spark = get_spark("analytics-spotify-user-ingestion")

## Read .json files and concatenate them into a single DataFrame

In [ ]:
files = glob.glob("../data/extended/Streaming_History_*.json")
dfs = [pd.read_json(file) for file in files]
df = pd.concat(dfs, ignore_index=True)

## Convert columns to appropriate data types

In [ ]:
df["ts"] = pd.to_datetime(df["ts"], utc=True)
df["audiobook_title"] = df["audiobook_title"].astype(str)
df["audiobook_uri"] = df["audiobook_uri"].astype(str)
df["audiobook_chapter_uri"] = df["audiobook_chapter_uri"].astype(str)
df["audiobook_chapter_title"] = df["audiobook_chapter_title"].astype(str)
df["offline"] = df["offline"].astype(str)

In [ ]:
print(len(df))
print(df.memory_usage(deep=True).sum() / 1024 / 1024, "MB")

## Create DataFrame

In [ ]:
df_spark = spark.createDataFrame(df)

## Save DataFrame

In [ ]:
if cfg["environment"] == "Local":
    print("✅ Environment: Local")
    
    # Export dataframe to CSV
    df_spark.toPandas().to_csv("../data/export/bronze/streaming_history.csv", index=False)
    print("✅ Exported CSV to ../data/export/bronze/streaming_history.csv")

elif cfg["environment"] == "Databricks":
    print("✅ Environment: Databricks")

    # Write to Delta Lake table
    df_spark.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(cfg["name_table_bronze"])
    print(f"✅ Loaded table: {cfg['name_table_bronze']}")

In [ ]:
spark.stop()